# 06 - Out-of-Sample Evaluation

This notebook evaluates the GARCH-family variance forecasts out of sample using the model specifications and custom MLE recursions from `03-GP-models-gaussian.ipynb`, `03-GP-models-student.ipynb`, and `04-extension-models.ipynb`. The recursion and likelihood functions are copied here so the notebook is self-contained.

Artifacts are written to `REPORTS/oos/` after each section. If an expensive artifact already exists and `FORCE_RECOMPUTE = False`, the notebook loads it instead of recomputing it.


In [ ]:
import json
import math
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.optimize import minimize
from scipy.special import gammaln
from scipy.stats import norm

warnings.filterwarnings("ignore", category=RuntimeWarning)

NOTEBOOK_START = time.perf_counter()

DATA_PATH = Path("CLEANED DATA/REMX_prices_sentiment_combined.xlsx")
OOS_DIR = Path("REPORTS/oos")
OOS_DIR.mkdir(parents=True, exist_ok=True)

RV_PATH = OOS_DIR / "rv_proxies.parquet"
RV_PLOT_PATH = OOS_DIR / "rv_series.png"
LEAD_LAG_PLOT_PATH = OOS_DIR / "lead_lag_diagnostic.png"
LEAD_LAG_CSV_PATH = OOS_DIR / "lead_lag_correlations.csv"
FORECASTS_PATH = OOS_DIR / "forecasts.parquet"
ROLLING_DIAG_PATH = OOS_DIR / "rolling_diagnostics.csv"
LOSSES_CSV_PATH = OOS_DIR / "losses.csv"
LOSS_OBS_PATH = OOS_DIR / "loss_observations.parquet"
COMPARISON_TESTS_PATH = OOS_DIR / "comparison_tests.csv"
HEADLINE_MD_PATH = OOS_DIR / "headline_results.md"

FORCE_RECOMPUTE = False

INSAMPLE_START = pd.Timestamp("2015-04-01")
INSAMPLE_END = pd.Timestamp("2022-12-31")
OOS_START = pd.Timestamp("2023-01-01")
OOS_END = pd.Timestamp("2026-03-31")
WINDOW_DAYS = 5 * 252
HORIZONS = [1, 5, 22]
VAR_FLOOR = 1e-8

MODEL_ORDER = [
    "GARCH(1,1)",
    "GARCH-X (tone)",
    "GARCH-X (art_growth)",
    "GARCHAND (tone)",
    "GARCHAND (art_growth)",
    "GARCHND (tone, kappa=30%)",
    "GARCHND (art_growth, kappa=30%)",
    "EGARCH-X (tone + art_growth)",
]
BASELINE_MODEL = "GARCH(1,1)"
RV_PROXY_MAP = {
    "Parkinson": "parkinson",
    "Rogers-Satchell": "rogers_satchell",
}

plt.style.use("seaborn-v0_8-whitegrid")


def load_clean_data():
    df = pd.read_excel(DATA_PATH, parse_dates=["Date"]).sort_values("Date").set_index("Date")
    required = [
        "Open", "High", "Low", "Close",
        "tone_mean_x100_winsor", "art_growth_winsor",
    ]
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in {DATA_PATH}: {missing}")

    df["r_raw"] = np.log(df["Close"] / df["Close"].shift(1))
    insample_mean = df.loc[INSAMPLE_START:INSAMPLE_END, "r_raw"].mean()
    df["r"] = df["r_raw"] - insample_mean
    df["r_pct"] = df["r"] * 100.0
    return df


def forward_sum(series, horizon):
    # Forward cumulative sum over date t through t+h-1.
    out = None
    for step in range(horizon):
        shifted = series.shift(-step)
        out = shifted if out is None else out + shifted
    valid_count = sum(series.shift(-step).notna().astype(int) for step in range(horizon))
    return out.where(valid_count == horizon)


df = load_clean_data()
print(f"Loaded {len(df):,} rows from {DATA_PATH}")
print(f"Sample span: {df.index.min().date()} to {df.index.max().date()}")
print(f"In-sample mean log return used for demeaning: {df.loc[INSAMPLE_START:INSAMPLE_END, 'r_raw'].mean():.6e}")


## Section 1 - Realized Variance Proxies

Compute Parkinson and Rogers-Satchell realized variance proxies from daily OHLC data. Both proxies are reported in percent-squared units so they match the percent-scale returns used in the GARCH estimation.


In [ ]:
if RV_PATH.exists() and not FORCE_RECOMPUTE:
    rv_df = pd.read_parquet(RV_PATH)
    if not isinstance(rv_df.index, pd.DatetimeIndex):
        rv_df.index = pd.to_datetime(rv_df.index)
    print(f"Loaded RV proxies from {RV_PATH}")
else:
    high = df["High"].astype(float)
    low = df["Low"].astype(float)
    open_ = df["Open"].astype(float)
    close = df["Close"].astype(float)

    positive_ohlc = (high > 0) & (low > 0) & (open_ > 0) & (close > 0) & (high >= low)
    parkinson = pd.Series(np.nan, index=df.index, name="parkinson")
    rogers_satchell = pd.Series(np.nan, index=df.index, name="rogers_satchell")

    parkinson.loc[positive_ohlc] = (
        (np.log(high.loc[positive_ohlc] / low.loc[positive_ohlc]) ** 2)
        / (4.0 * np.log(2.0))
        * 10000.0
    )
    rogers_satchell.loc[positive_ohlc] = (
        np.log(high.loc[positive_ohlc] / close.loc[positive_ohlc])
        * np.log(high.loc[positive_ohlc] / open_.loc[positive_ohlc])
        + np.log(low.loc[positive_ohlc] / close.loc[positive_ohlc])
        * np.log(low.loc[positive_ohlc] / open_.loc[positive_ohlc])
    ) * 10000.0

    negative_rs = rogers_satchell < 0
    n_negative_rs = int(negative_rs.sum())
    if n_negative_rs:
        rogers_satchell.loc[negative_rs] = np.nan
    print(f"Rogers-Satchell negative values set to NaN: {n_negative_rs}")

    rv_df = pd.concat([parkinson, rogers_satchell], axis=1)
    rv_df.to_parquet(RV_PATH)
    print(f"Saved RV proxies to {RV_PATH}")

mean_park = rv_df["parkinson"].mean()
return_var = df["r_pct"].dropna().var()
ratio = mean_park / return_var
print(f"Mean Parkinson RV (%^2): {mean_park:.4f}")
print(f"Unconditional variance of percent returns: {return_var:.4f}")
print(f"Parkinson / return-variance ratio: {ratio:.4f}")
assert 0.1 <= ratio <= 10.0, "Parkinson proxy and return variance differ by more than 10x."

fig, ax = plt.subplots(figsize=(13, 5))
rv_df["parkinson"].plot(ax=ax, lw=0.9, alpha=0.85, label="Parkinson")
rv_df["rogers_satchell"].plot(ax=ax, lw=0.8, alpha=0.75, label="Rogers-Satchell")

events = {
    "COVID\nMar 2020": pd.Timestamp("2020-03-01"),
    "Russia-Ukraine\nFeb 2022": pd.Timestamp("2022-02-24"),
    "China export controls\nApr 2025": pd.Timestamp("2025-04-01"),
}
for label, date in events.items():
    ax.axvline(date, color="black", ls="--", lw=1.0, alpha=0.65)
    ax.text(date, ax.get_ylim()[1] * 0.95, label, rotation=90, va="top", ha="right", fontsize=8)

ax.set_title("Realized variance proxies from OHLC data")
ax.set_ylabel("Daily realized variance (%^2)")
ax.set_xlabel("")
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(RV_PLOT_PATH, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved RV plot to {RV_PLOT_PATH}")


## Section 2 - Lead-Lag Diagnostic

For lags `k = -10, ..., +10`, compute correlations between sentiment at date `t` and Parkinson realized variance at date `t+k`. Negative lag means sentiment lags volatility; positive lag means sentiment leads volatility. Peak lags below are selected by the strongest absolute correlation, so the sign of `k` gives the lead-lag direction.


In [ ]:
rv_df = pd.read_parquet(RV_PATH)
rv_df.index = pd.to_datetime(rv_df.index)

lead_lag_sources = {
    "tone_mean_x100_winsor": df["tone_mean_x100_winsor"],
    "abs_tone_mean_x100_winsor": df["tone_mean_x100_winsor"].abs(),
    "art_growth_winsor": df["art_growth_winsor"],
}
lags = list(range(-10, 11))
rows = []
for series_name, series in lead_lag_sources.items():
    for lag in lags:
        corr = series.corr(rv_df["parkinson"].shift(-lag))
        rows.append({"series": series_name, "lag": lag, "correlation": corr})

lead_lag_df = pd.DataFrame(rows)
lead_lag_df.to_csv(LEAD_LAG_CSV_PATH, index=False)

peak_rows = []
for series_name, group in lead_lag_df.groupby("series", sort=False):
    idx = group["correlation"].abs().idxmax()
    peak = group.loc[idx]
    peak_rows.append(peak)
    direction = "leads volatility" if peak["lag"] > 0 else "lags volatility" if peak["lag"] < 0 else "is contemporaneous with volatility"
    print(
        f"{series_name}: strongest absolute correlation at k={int(peak['lag']):+d} "
        f"({direction}), corr={peak['correlation']:.4f}"
    )
peak_df = pd.DataFrame(peak_rows).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 5))
for series_name, group in lead_lag_df.groupby("series", sort=False):
    ax.plot(group["lag"], group["correlation"], marker="o", lw=1.6, label=series_name)
ax.axvline(0, color="black", ls="--", lw=1.0)
ax.axhline(0, color="black", lw=0.8, alpha=0.45)
ax.set_xticks(lags)
ax.set_xlabel("Lag k: corr(sentiment_t, Parkinson_{t+k})")
ax.set_ylabel("Correlation")
ax.set_title("Lead-lag correlations between sentiment and realized volatility")
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(LEAD_LAG_PLOT_PATH, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved lead-lag plot to {LEAD_LAG_PLOT_PATH}")


### Lead-lag interpretation

- `tone_mean_x100_winsor` has its strongest absolute correlation at `k=-3` with correlation `0.0799`; this means tone lags realized volatility, which weakens the case that this tone signal forecasts volatility.
- `abs_tone_mean_x100_winsor` has its strongest absolute correlation at `k=+10` with correlation `-0.0421`; the positive lag means absolute tone leads realized volatility, which is directionally consistent with a longer-horizon sentiment channel, although the correlation is negative and small.
- `art_growth_winsor` has its strongest absolute correlation at `k=-2` with correlation `-0.0204`; this means article-growth changes lag realized volatility, which weakens the forecasting interpretation for this variable.


## Section 3 - Rolling Forecast Loop

This section fits each model on a trailing five-year window ending at `t-1`, then forecasts cumulative conditional variance for horizons `h = 1, 5, 22` beginning at target date `t`. Student-t innovations are used throughout.


### Forecasting Conventions

For GARCH-family multi-step forecasts, the expected squared return in forecasted steps is the model's own previous variance forecast. For exogenous sentiment variables, the first step uses the last observed value in the training window, because it is known at the forecast origin `t-1`; later steps use the trailing-window mean because future sentiment is unknown.

For EGARCH-X, the first step uses the last fitted standardized residual. For forecasted steps beyond the first, the standardized shock term `|z| - E|z|` and leverage term `z` are set to their unconditional means of zero under symmetric Student-t innovations, so only persistence and exogenous terms propagate.


In [ ]:
# Student-t likelihood and model recursions copied from the in-sample notebooks.
LARGE = 1e12
SMOOTH_K = 20.0
CLAMP = 30.0


def _t_logpdf(z, nu):
    if nu <= 2.0 or not np.isfinite(nu):
        return np.full_like(z, -np.inf, dtype=float)
    const = (
        gammaln((nu + 1.0) / 2.0)
        - gammaln(nu / 2.0)
        - 0.5 * np.log(np.pi * (nu - 2.0))
    )
    return const - 0.5 * (nu + 1.0) * np.log1p((z ** 2) / (nu - 2.0))


def _et_abs_z(nu):
    if nu <= 2.0 or not np.isfinite(nu):
        return np.nan
    return (
        2.0
        * np.sqrt(nu - 2.0)
        * np.exp(gammaln((nu + 1.0) / 2.0) - gammaln(nu / 2.0))
        / ((nu - 1.0) * np.sqrt(np.pi))
    )


def d3_smooth(z):
    z = np.clip(SMOOTH_K * z, -60.0, 60.0)
    return 1.0 / (1.0 + np.exp(-z))


def annual_vol_to_daily_var(v_ann_pct):
    return (v_ann_pct / np.sqrt(252.0)) ** 2


def params_to_array(spec, params):
    return np.array([params[name] for name in spec["param_names"]], dtype=float)


def array_to_params(spec, theta):
    return {name: float(value) for name, value in zip(spec["param_names"], theta)}


def in_bounds(theta, bounds):
    for value, (lo, hi) in zip(theta, bounds):
        if not np.isfinite(value):
            return False
        if lo is not None and value < lo:
            return False
        if hi is not None and value > hi:
            return False
    return True


def project_bounds(theta, bounds, eps=1e-10):
    out = np.asarray(theta, dtype=float).copy()
    for i, (lo, hi) in enumerate(bounds):
        if lo is not None:
            out[i] = max(out[i], lo + eps)
        if hi is not None:
            out[i] = min(out[i], hi - eps)
    return out


def garch_per_obs_ll(theta, r, x=None, variant="baseline", kappa=None):
    p = dict(zip(MODEL_PARAM_NAMES[variant], theta))
    nu = p["nu"]
    if nu <= 2.05:
        return None, None

    n = len(r)
    sig2 = np.empty(n, dtype=float)
    sig2[0] = max(float(np.nanvar(r)), VAR_FLOOR)

    for t in range(1, n):
        news = 0.0
        if variant == "garchx":
            news = p["gamma"] * x[t - 1] ** 2
        elif variant == "garchand_tone":
            d1 = 1.0 if x[t - 1] < 0.0 else 0.0
            news = p["gamma"] * (1.0 + d1 * p["theta"]) * x[t - 1] ** 2
        elif variant == "garchand_art":
            d2 = 1.0 if x[t - 1] > 0.0 else 0.0
            news = p["gamma"] * d2 * x[t - 1] ** 2
        elif variant == "garchnd":
            d3 = d3_smooth(sig2[t - 1] - kappa)
            news = p["gamma"] * d3 * x[t - 1] ** 2
        elif variant == "baseline":
            news = 0.0
        else:
            raise ValueError(f"Unknown GARCH variant: {variant}")

        s = p["omega"] + p["alpha"] * r[t - 1] ** 2 + p["beta"] * sig2[t - 1] + news
        sig2[t] = s if np.isfinite(s) and s > VAR_FLOOR else VAR_FLOOR

    z = r / np.sqrt(sig2)
    ll_t = _t_logpdf(z, nu) - 0.5 * np.log(sig2)
    return ll_t, sig2


def egarch_per_obs_ll(theta, r, tone, art_growth):
    mu, omega, alpha, xi, beta, g1, g2, nu = theta
    if nu <= 2.05:
        return None, None

    n = len(r)
    eps = r - mu
    log_sig2 = np.empty(n, dtype=float)
    log_sig2[0] = np.log(max(float(np.nanvar(eps)), VAR_FLOOR))
    e_abs_z = _et_abs_z(nu)
    if not np.isfinite(e_abs_z):
        return None, None

    for t in range(1, n):
        z_prev = eps[t - 1] / np.exp(0.5 * log_sig2[t - 1])
        ls = (
            omega
            + alpha * (abs(z_prev) - e_abs_z)
            + xi * z_prev
            + beta * log_sig2[t - 1]
            + g1 * tone[t - 1]
            + g2 * art_growth[t - 1]
        )
        log_sig2[t] = np.clip(ls, -CLAMP, CLAMP) if np.isfinite(ls) else np.log(VAR_FLOOR)

    sig2 = np.exp(log_sig2)
    z2 = eps ** 2 / sig2
    log_const = (
        gammaln((nu + 1.0) / 2.0)
        - gammaln(nu / 2.0)
        - 0.5 * np.log(np.pi * (nu - 2.0))
    )
    ll_t = log_const - 0.5 * log_sig2 - ((nu + 1.0) / 2.0) * np.log1p(z2 / (nu - 2.0))
    return ll_t, log_sig2


MODEL_PARAM_NAMES = {
    "baseline": ["omega", "alpha", "beta", "nu"],
    "garchx": ["omega", "alpha", "beta", "gamma", "nu"],
    "garchand_tone": ["omega", "alpha", "beta", "gamma", "theta", "nu"],
    "garchand_art": ["omega", "alpha", "beta", "gamma", "nu"],
    "garchnd": ["omega", "alpha", "beta", "gamma", "nu"],
    "egarchx": ["mu", "omega", "alpha", "xi", "beta", "gamma_tone", "gamma_artgrowth", "nu"],
}


def neg_loglik(theta, spec, train):
    if not in_bounds(theta, spec["bounds"]):
        return LARGE
    try:
        if spec["family"] == "egarchx":
            ll_t, _ = egarch_per_obs_ll(
                theta,
                train["r"].to_numpy(dtype=float),
                train["tone_mean_x100_winsor"].to_numpy(dtype=float),
                train["art_growth_winsor"].to_numpy(dtype=float),
            )
        else:
            x = None
            if spec.get("exog_cols"):
                x = train[spec["exog_cols"][0]].to_numpy(dtype=float)
            ll_t, _ = garch_per_obs_ll(
                theta,
                train["r"].to_numpy(dtype=float),
                x=x,
                variant=spec["family"],
                kappa=spec.get("kappa"),
            )
        if ll_t is None or not np.all(np.isfinite(ll_t)):
            return LARGE
        value = -float(np.sum(ll_t))
        return value if np.isfinite(value) else LARGE
    except Exception:
        return LARGE


def fit_one_window(spec, train, warm_theta):
    warm_theta = project_bounds(warm_theta, spec["bounds"])
    objective = lambda theta: neg_loglik(theta, spec, train)

    opt = minimize(
        objective,
        warm_theta,
        method="L-BFGS-B",
        bounds=spec["bounds"],
        options={"maxiter": 2000, "ftol": 1e-8, "maxls": 50},
    )
    if opt.success and np.isfinite(opt.fun):
        return project_bounds(opt.x, spec["bounds"]), True, "L-BFGS-B"

    nm_objective = lambda theta: objective(theta) if in_bounds(theta, spec["bounds"]) else LARGE
    opt_nm = minimize(
        nm_objective,
        warm_theta,
        method="Nelder-Mead",
        options={"maxiter": 4000, "xatol": 1e-7, "fatol": 1e-7},
    )
    if opt_nm.success and np.isfinite(opt_nm.fun) and opt_nm.fun < LARGE:
        return project_bounds(opt_nm.x, spec["bounds"]), True, "Nelder-Mead"

    return warm_theta, False, "carry-forward"


In [ ]:
# Hard-coded warm starts from the in-sample notebooks/reports.
KAPPA_30 = annual_vol_to_daily_var(30.0)

INITIAL_PARAMS = {
    "GARCH(1,1)": {
        "omega": 0.0876, "alpha": 0.0840, "beta": 0.9010, "nu": 11.1687,
    },
    "GARCH-X (tone)": {
        "omega": 0.1202, "alpha": 0.0843, "beta": 0.8984, "gamma": -0.0179, "nu": 10.8955,
    },
    "GARCH-X (art_growth)": {
        "omega": 0.1770, "alpha": 0.0989, "beta": 0.8872, "gamma": -0.1067, "nu": 5.4352,
    },
    "GARCHAND (tone)": {
        "omega": 0.1252, "alpha": 0.0839, "beta": 0.8976, "gamma": -0.0093, "theta": 1.2679, "nu": 11.0445,
    },
    "GARCHAND (art_growth)": {
        "omega": 0.0892, "alpha": 0.0852, "beta": 0.8996, "gamma": 0.0, "nu": 11.1147,
    },
    "GARCHND (tone, kappa=30%)": {
        "omega": 0.0896, "alpha": 0.0852, "beta": 0.8994, "gamma": 0.0012, "nu": 11.1313,
    },
    "GARCHND (art_growth, kappa=30%)": {
        "omega": 0.0787, "alpha": 0.0759, "beta": 0.9149, "gamma": -0.0679, "nu": 10.3328,
    },
    "EGARCH-X (tone + art_growth)": {
        "mu": -0.0240, "omega": 0.0243, "alpha": 0.1532, "xi": -0.0142,
        "beta": 0.9779, "gamma_tone": 0.0001, "gamma_artgrowth": -0.0921, "nu": 8.0046,
    },
}

MODEL_SPECS = [
    {
        "name": "GARCH(1,1)",
        "family": "baseline",
        "param_names": MODEL_PARAM_NAMES["baseline"],
        "bounds": [(1e-8, None), (0.0, 0.999), (0.0, 0.999), (2.05, 100.0)],
        "exog_cols": [],
    },
    {
        "name": "GARCH-X (tone)",
        "family": "garchx",
        "param_names": MODEL_PARAM_NAMES["garchx"],
        "bounds": [(1e-8, None), (0.0, 0.999), (0.0, 0.999), (None, None), (2.05, 100.0)],
        "exog_cols": ["tone_mean_x100_winsor"],
    },
    {
        "name": "GARCH-X (art_growth)",
        "family": "garchx",
        "param_names": MODEL_PARAM_NAMES["garchx"],
        "bounds": [(1e-8, None), (0.0, 0.999), (0.0, 0.999), (None, None), (2.05, 100.0)],
        "exog_cols": ["art_growth_winsor"],
    },
    {
        "name": "GARCHAND (tone)",
        "family": "garchand_tone",
        "param_names": MODEL_PARAM_NAMES["garchand_tone"],
        "bounds": [(1e-8, None), (0.0, 0.999), (0.0, 0.999), (None, None), (-0.999, None), (2.05, 100.0)],
        "exog_cols": ["tone_mean_x100_winsor"],
    },
    {
        "name": "GARCHAND (art_growth)",
        "family": "garchand_art",
        "param_names": MODEL_PARAM_NAMES["garchand_art"],
        "bounds": [(1e-8, None), (0.0, 0.999), (0.0, 0.999), (None, None), (2.05, 100.0)],
        "exog_cols": ["art_growth_winsor"],
    },
    {
        "name": "GARCHND (tone, kappa=30%)",
        "family": "garchnd",
        "param_names": MODEL_PARAM_NAMES["garchnd"],
        "bounds": [(1e-8, None), (0.0, 0.999), (0.0, 0.999), (None, None), (2.05, 100.0)],
        "exog_cols": ["tone_mean_x100_winsor"],
        "kappa": KAPPA_30,
    },
    {
        "name": "GARCHND (art_growth, kappa=30%)",
        "family": "garchnd",
        "param_names": MODEL_PARAM_NAMES["garchnd"],
        "bounds": [(1e-8, None), (0.0, 0.999), (0.0, 0.999), (None, None), (2.05, 100.0)],
        "exog_cols": ["art_growth_winsor"],
        "kappa": KAPPA_30,
    },
    {
        "name": "EGARCH-X (tone + art_growth)",
        "family": "egarchx",
        "param_names": MODEL_PARAM_NAMES["egarchx"],
        "bounds": [(None, None), (None, None), (None, None), (None, None), (-0.999, 0.999), (None, None), (None, None), (2.05, 200.0)],
        "exog_cols": ["tone_mean_x100_winsor", "art_growth_winsor"],
    },
]

MODEL_ORDER = [spec["name"] for spec in MODEL_SPECS]
BASELINE_MODEL = "GARCH(1,1)"
print(f"kappa for 30% annual volatility: {KAPPA_30:.4f} percent^2/day")


In [ ]:
ROLLING_DIAGNOSTICS = {}


def _one_step_garch_forecast(spec, p, prev_sig2, prev_r2, x_value=None):
    family = spec["family"]
    news = 0.0
    if family == "garchx":
        news = p["gamma"] * x_value ** 2
    elif family == "garchand_tone":
        d1 = 1.0 if x_value < 0.0 else 0.0
        news = p["gamma"] * (1.0 + d1 * p["theta"]) * x_value ** 2
    elif family == "garchand_art":
        d2 = 1.0 if x_value > 0.0 else 0.0
        news = p["gamma"] * d2 * x_value ** 2
    elif family == "garchnd":
        d3 = d3_smooth(prev_sig2 - spec["kappa"])
        news = p["gamma"] * d3 * x_value ** 2
    elif family == "baseline":
        news = 0.0
    else:
        raise ValueError(f"Unknown GARCH family: {family}")
    return p["omega"] + p["alpha"] * prev_r2 + p["beta"] * prev_sig2 + news


def forecast_cumulative_variance(spec, theta, train, horizons):
    max_h = max(horizons)
    floor_count = 0
    p = array_to_params(spec, theta)
    r = train["r"].to_numpy(dtype=float)

    if spec["family"] == "egarchx":
        tone = train["tone_mean_x100_winsor"].to_numpy(dtype=float)
        art = train["art_growth_winsor"].to_numpy(dtype=float)
        _, log_sig2 = egarch_per_obs_ll(theta, r, tone, art)
        if log_sig2 is None:
            raise ValueError("EGARCH recursion failed during forecasting.")
        prev_log_sig2 = float(log_sig2[-1])
        mu, omega, alpha, xi, beta, g1, g2, nu = theta
        e_abs_z = _et_abs_z(nu)
        prev_z = (r[-1] - mu) / np.exp(0.5 * prev_log_sig2)
        tone_mean = float(np.nanmean(tone))
        art_mean = float(np.nanmean(art))

        step_vars = []
        for step in range(1, max_h + 1):
            if step == 1:
                shock_term = abs(prev_z) - e_abs_z
                leverage_term = prev_z
                tone_value = float(tone[-1])
                art_value = float(art[-1])
            else:
                shock_term = 0.0
                leverage_term = 0.0
                tone_value = tone_mean
                art_value = art_mean
            ls = (
                omega
                + alpha * shock_term
                + xi * leverage_term
                + beta * prev_log_sig2
                + g1 * tone_value
                + g2 * art_value
            )
            v = float(np.exp(np.clip(ls, -CLAMP, CLAMP))) if np.isfinite(ls) else VAR_FLOOR
            if v <= VAR_FLOOR:
                floor_count += 1
                v = VAR_FLOOR
            step_vars.append(v)
            prev_log_sig2 = math.log(max(v, VAR_FLOOR))
        return {h: float(np.sum(step_vars[:h])) for h in horizons}, floor_count

    x = None
    x_mean = None
    if spec.get("exog_cols"):
        x = train[spec["exog_cols"][0]].to_numpy(dtype=float)
        x_mean = float(np.nanmean(x))
    _, sig2 = garch_per_obs_ll(theta, r, x=x, variant=spec["family"], kappa=spec.get("kappa"))
    if sig2 is None:
        raise ValueError("GARCH recursion failed during forecasting.")

    prev_sig2 = float(sig2[-1])
    prev_r2 = float(r[-1] ** 2)
    step_vars = []
    for step in range(1, max_h + 1):
        x_value = None
        if x is not None:
            x_value = float(x[-1]) if step == 1 else x_mean
        v = _one_step_garch_forecast(spec, p, prev_sig2, prev_r2, x_value=x_value)
        if not np.isfinite(v) or v <= VAR_FLOOR:
            floor_count += 1
            v = VAR_FLOOR
        v = float(v)
        step_vars.append(v)
        prev_sig2 = v
        prev_r2 = v
    return {h: float(np.sum(step_vars[:h])) for h in horizons}, floor_count


def rolling_forecast(model_spec, returns, exog_dict, oos_dates, window_days, horizons):
    model_name = model_spec["name"]
    needed = [returns.rename("r")]
    for col in model_spec.get("exog_cols", []):
        needed.append(exog_dict[col].rename(col))
    data = pd.concat(needed, axis=1).dropna()
    data = data.sort_index()

    usable_dates = [pd.Timestamp(d) for d in oos_dates if pd.Timestamp(d) in data.index]
    warm_theta = params_to_array(model_spec, INITIAL_PARAMS[model_name])
    last_good_theta = project_bounds(warm_theta, model_spec["bounds"])
    rows = []
    failures = 0
    floor_events = 0
    methods = {"L-BFGS-B": 0, "Nelder-Mead": 0, "carry-forward": 0}

    print(f"\nStarting rolling loop for {model_name}: {len(usable_dates)} OOS target dates")
    for i, target_date in enumerate(usable_dates):
        loc = data.index.get_loc(target_date)
        start = max(0, loc - window_days)
        train = data.iloc[start:loc]
        if len(train) < max(250, len(model_spec["param_names"]) * 20):
            continue

        theta_hat, converged, method = fit_one_window(model_spec, train, last_good_theta)
        methods[method] = methods.get(method, 0) + 1
        if converged:
            last_good_theta = theta_hat
        else:
            failures += 1
            theta_hat = last_good_theta

        cum_forecasts, floors = forecast_cumulative_variance(model_spec, theta_hat, train, horizons)
        floor_events += floors
        params_dict = array_to_params(model_spec, theta_hat)
        for horizon in horizons:
            rows.append(
                {
                    "date": target_date,
                    "model": model_name,
                    "horizon": int(horizon),
                    "variance_forecast": max(float(cum_forecasts[horizon]), VAR_FLOOR),
                    "converged": bool(converged),
                    "params": params_dict,
                }
            )

        if (i + 1) % 50 == 0 or i == 0 or (i + 1) == len(usable_dates):
            print(
                f"{model_name}: processed {i + 1}/{len(usable_dates)} "
                f"through {target_date.date()} | failures={failures} | forecast floors={floor_events}"
            )

    out = pd.DataFrame(rows)
    ROLLING_DIAGNOSTICS[model_name] = {
        "failures": failures,
        "forecast_floor_events": floor_events,
        "n_rows": len(out),
        **{f"method_{k}": v for k, v in methods.items()},
    }
    print(
        f"Completed {model_name}: failures={failures}, "
        f"forecast flooring events={floor_events}, rows={len(out)}"
    )
    return out


In [ ]:
returns = df["r_pct"].dropna()
exog_dict = {
    "tone_mean_x100_winsor": df["tone_mean_x100_winsor"],
    "art_growth_winsor": df["art_growth_winsor"],
}
oos_dates = returns.loc[OOS_START:OOS_END].index
print(f"OOS target dates requested: {OOS_START.date()} to {OOS_END.date()}")
print(f"Actual OOS trading dates: {oos_dates.min().date()} to {oos_dates.max().date()} ({len(oos_dates)} dates)")
print(f"Rolling estimation window: {WINDOW_DAYS} trading days")

if FORECASTS_PATH.exists() and not FORCE_RECOMPUTE:
    forecasts = pd.read_parquet(FORECASTS_PATH)
    forecasts["date"] = pd.to_datetime(forecasts["date"])
    print(f"Loaded existing forecasts from {FORECASTS_PATH}: {forecasts.shape}")
else:
    dry_specs = [MODEL_SPECS[0], MODEL_SPECS[1]]
    dry_dates = oos_dates[:50]
    print("Running dry run: baseline GARCH(1,1) and GARCH-X(tone), first 50 OOS dates.")
    dry_forecasts = pd.concat(
        [
            rolling_forecast(spec, returns, exog_dict, dry_dates, WINDOW_DAYS, HORIZONS)
            for spec in dry_specs
        ],
        ignore_index=True,
    )
    display(dry_forecasts)

    assert np.isfinite(dry_forecasts["variance_forecast"]).all(), "Dry-run forecasts contain non-finite values."
    assert (dry_forecasts["variance_forecast"] > 0).all(), "Dry-run forecasts must be positive."
    h_means = dry_forecasts.groupby(["model", "horizon"])["variance_forecast"].mean().unstack()
    display(h_means)
    h5_to_h1 = h_means[5] / h_means[1]
    print("Dry-run mean h=5 / h=1 cumulative forecast ratios:")
    print(h5_to_h1.to_string(float_format=lambda x: f"{x:.3f}"))
    assert (h5_to_h1 > 3.0).all(), "Dry-run h=5 forecasts are not materially larger than h=1 forecasts."
    assert (h5_to_h1 < 8.0).all(), "Dry-run h=5/h=1 ratios look implausibly large."

    q = dry_forecasts.groupby("horizon")["variance_forecast"].quantile([0.01, 0.50, 0.99]).unstack()
    print("Dry-run forecast quantiles by horizon:")
    display(q)
    assert dry_forecasts["variance_forecast"].quantile(0.99) < 1000.0, "Dry-run forecasts are outside a broad plausible range."

    print("Dry-run sanity checks passed. Running the full eight-model grid.")
    ROLLING_DIAGNOSTICS.clear()
    forecasts = pd.concat(
        [
            rolling_forecast(spec, returns, exog_dict, oos_dates, WINDOW_DAYS, HORIZONS)
            for spec in MODEL_SPECS
        ],
        ignore_index=True,
    )
    forecasts.to_parquet(FORECASTS_PATH, index=False)
    pd.DataFrame.from_dict(ROLLING_DIAGNOSTICS, orient="index").to_csv(ROLLING_DIAG_PATH)
    print(f"Saved full forecasts to {FORECASTS_PATH}")
    print(f"Saved rolling diagnostics to {ROLLING_DIAG_PATH}")

display(forecasts.head())
print(forecasts.groupby(["model", "horizon"])["variance_forecast"].agg(["count", "mean", "min", "max"]).to_string())


## Section 4 - Loss Functions

Load the forecast and realized-variance artifacts, align each forecast with cumulative realized variance over the matching horizon, and compute RMSE, MAE, HRMSE, and QLIKE for Parkinson and Rogers-Satchell proxies.


In [ ]:
forecasts = pd.read_parquet(FORECASTS_PATH)
forecasts["date"] = pd.to_datetime(forecasts["date"])
rv_df = pd.read_parquet(RV_PATH)
rv_df.index = pd.to_datetime(rv_df.index)

rv_proxy_map = RV_PROXY_MAP.copy()

loss_rows = []
summary_rows = []
for proxy_label, proxy_col in rv_proxy_map.items():
    rv_series = rv_df[proxy_col]
    for horizon in HORIZONS:
        rv_cum = forward_sum(rv_series, horizon).rename("realized_variance")
        sub = forecasts.loc[forecasts["horizon"] == horizon].copy()
        sub = sub.merge(rv_cum, left_on="date", right_index=True, how="left")
        sub["rv_proxy"] = proxy_label
        sub["forecast_floor"] = sub["variance_forecast"].clip(lower=VAR_FLOOR)
        valid = sub["realized_variance"].gt(0) & sub["forecast_floor"].gt(0)
        sub.loc[~valid, ["realized_variance", "forecast_floor"]] = np.nan
        err = sub["realized_variance"] - sub["forecast_floor"]
        sub["SE"] = err ** 2
        sub["AE"] = err.abs()
        sub["HRMSE_component"] = (err / sub["realized_variance"]) ** 2
        ratio = sub["realized_variance"] / sub["forecast_floor"]
        sub["QLIKE_component"] = ratio - np.log(ratio) - 1.0
        loss_rows.append(
            sub[
                [
                    "date", "model", "horizon", "rv_proxy", "realized_variance",
                    "variance_forecast", "SE", "AE", "HRMSE_component", "QLIKE_component",
                ]
            ]
        )

        for model, grp in sub.groupby("model", sort=False):
            clean = grp.dropna(subset=["SE", "AE", "HRMSE_component", "QLIKE_component"])
            summary_rows.append(
                {
                    "model": model,
                    "horizon": horizon,
                    "rv_proxy": proxy_label,
                    "RMSE": float(np.sqrt(clean["SE"].mean())),
                    "MAE": float(clean["AE"].mean()),
                    "HRMSE": float(np.sqrt(clean["HRMSE_component"].mean())),
                    "QLIKE": float(clean["QLIKE_component"].mean()),
                    "n": int(len(clean)),
                }
            )

loss_obs = pd.concat(loss_rows, ignore_index=True)
loss_obs.to_parquet(LOSS_OBS_PATH, index=False)

loss_long = pd.DataFrame(summary_rows)
loss_summary = loss_long.pivot(index="model", columns=["horizon", "rv_proxy"], values=["RMSE", "MAE", "HRMSE", "QLIKE"])
loss_summary = loss_summary.reorder_levels([1, 2, 0], axis=1).sort_index(axis=1)
loss_summary = loss_summary.reindex(MODEL_ORDER)
loss_summary.to_csv(LOSSES_CSV_PATH)
print(f"Saved full loss summary to {LOSSES_CSV_PATH}")
print(f"Saved per-observation loss panel to {LOSS_OBS_PATH}")

headline_cols = pd.MultiIndex.from_product([HORIZONS, ["Parkinson"], ["HRMSE", "QLIKE"]])
headline = loss_summary.loc[:, loss_summary.columns.intersection(headline_cols)]
display(Markdown("### Headline loss table: Parkinson HRMSE and QLIKE"))
display(headline.style.format("{:.4f}").highlight_min(axis=0, color="#d7f0d2"))

display(Markdown("### Full loss table"))
display(loss_summary.style.format("{:.4f}").highlight_min(axis=0, color="#d7f0d2"))


## Section 5 - Model Comparison Tests

Run Diebold-Mariano tests versus the GARCH(1,1) baseline, Clark-West tests for nested comparisons, and a Model Confidence Set at the 90% level across all eight models. Tests are run separately for each horizon and RV proxy.


In [ ]:
forecasts = pd.read_parquet(FORECASTS_PATH)
forecasts["date"] = pd.to_datetime(forecasts["date"])
rv_df = pd.read_parquet(RV_PATH)
rv_df.index = pd.to_datetime(rv_df.index)
rv_proxy_map = RV_PROXY_MAP.copy()


def newey_west_lrv(x, bandwidth):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    x = x - x.mean()
    n = len(x)
    if n <= 1:
        return np.nan
    gamma0 = float(np.dot(x, x) / n)
    lrv = gamma0
    for lag in range(1, bandwidth + 1):
        cov = float(np.dot(x[lag:], x[:-lag]) / n)
        weight = 1.0 - lag / (bandwidth + 1.0)
        lrv += 2.0 * weight * cov
    return max(lrv, 0.0)


def mean_t_test(x, bandwidth, alternative="two-sided"):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    n = len(x)
    if n <= 2:
        return np.nan, np.nan, n
    mean_x = float(np.mean(x))
    lrv = newey_west_lrv(x, bandwidth)
    se = math.sqrt(lrv / n) if np.isfinite(lrv) and lrv > 0 else np.nan
    stat = mean_x / se if se and np.isfinite(se) else np.nan
    if not np.isfinite(stat):
        return np.nan, np.nan, n
    if alternative == "greater":
        p_value = 1.0 - norm.cdf(stat)
    elif alternative == "less":
        p_value = norm.cdf(stat)
    else:
        p_value = 2.0 * (1.0 - norm.cdf(abs(stat)))
    return float(stat), float(p_value), n


def forecast_wide_for(horizon):
    sub = forecasts.loc[forecasts["horizon"] == horizon]
    wide = sub.pivot(index="date", columns="model", values="variance_forecast")
    return wide.reindex(columns=MODEL_ORDER)


def loss_matrix(horizon, proxy_label, loss_type):
    proxy_col = rv_proxy_map[proxy_label]
    rv_cum = forward_sum(rv_df[proxy_col], horizon).rename("rv")
    fwide = forecast_wide_for(horizon)
    panel = fwide.join(rv_cum, how="left").dropna(subset=["rv"])
    panel = panel.loc[panel["rv"] > 0].copy()
    forecast_values = panel[MODEL_ORDER].clip(lower=VAR_FLOOR)
    rv_values = panel["rv"]
    if loss_type == "SE":
        losses = forecast_values.sub(rv_values, axis=0) ** 2
    elif loss_type == "HRMSE":
        losses = (forecast_values.sub(rv_values, axis=0).div(rv_values, axis=0)) ** 2
    else:
        raise ValueError(loss_type)
    return losses.dropna(how="any"), rv_values.reindex(losses.index), forecast_values.reindex(losses.index)


def fallback_mcs(losses, alpha=0.10, reps=2000, seed=12345):
    rng = np.random.default_rng(seed)
    current = list(losses.columns)
    pvalues = {model: np.nan for model in current}
    while len(current) > 1:
        L = losses[current].dropna().to_numpy(dtype=float)
        n, m = L.shape
        means = L.mean(axis=0)
        centered = L - means
        se = centered.std(axis=0, ddof=1) / np.sqrt(n)
        se = np.where(se <= 0, np.nan, se)
        obs_stat = np.nanmax((means - means.mean()) / se)
        boot_stats = []
        for _ in range(reps):
            idx = rng.integers(0, n, size=n)
            sample = centered[idx, :]
            sample_means = sample.mean(axis=0)
            boot_stats.append(np.nanmax((sample_means - sample_means.mean()) / se))
        p_value = float(np.mean(np.asarray(boot_stats) >= obs_stat))
        if p_value > alpha:
            break
        worst = current[int(np.nanargmax(means))]
        pvalues[worst] = p_value
        current.remove(worst)
    for model in current:
        pvalues[model] = 1.0
    return current, pvalues, "fallback_centered_bootstrap"


def run_mcs(losses, horizon, alpha=0.10):
    losses = losses.dropna(how="any")
    try:
        from arch.bootstrap import MCS

        block_size = max(1, int(horizon))
        mcs = MCS(losses, size=alpha, reps=1000, block_size=block_size, method="R", seed=12345)
        mcs.compute()
        pvals = mcs.pvalues["Pvalue"].to_dict()
        return list(mcs.included), pvals, "arch.bootstrap.MCS"
    except Exception as exc:
        included, pvals, method = fallback_mcs(losses, alpha=alpha)
        print(f"MCS fallback used for h={horizon}: {exc}")
        return included, pvals, method


test_rows = []
for proxy_label in rv_proxy_map:
    for horizon in HORIZONS:
        bandwidth = horizon - 1
        for loss_type in ["SE", "HRMSE"]:
            losses, rv_values, fwide = loss_matrix(horizon, proxy_label, loss_type)
            base_loss = losses[BASELINE_MODEL]
            for model in MODEL_ORDER:
                if model == BASELINE_MODEL:
                    continue
                diff = base_loss - losses[model]
                stat, p_value, n_obs = mean_t_test(diff, bandwidth, alternative="two-sided")
                test_rows.append(
                    {
                        "test": "Diebold-Mariano",
                        "horizon": horizon,
                        "rv_proxy": proxy_label,
                        "loss_type": loss_type,
                        "model": model,
                        "statistic": stat,
                        "p_value": p_value,
                        "n": n_obs,
                        "bandwidth": bandwidth,
                        "null": "Equal predictive accuracy vs baseline",
                        "alternative": "two-sided; positive statistic favors augmented model",
                        "included_mcs": np.nan,
                        "method": "Newey-West" if bandwidth > 0 else "standard",
                    }
                )

        se_losses, rv_values, fwide = loss_matrix(horizon, proxy_label, "SE")
        f0 = fwide[BASELINE_MODEL]
        for model in MODEL_ORDER:
            if model == BASELINE_MODEL:
                continue
            f1 = fwide[model]
            cw = (rv_values - f0) ** 2 - ((rv_values - f1) ** 2 - (f0 - f1) ** 2)
            stat, p_value, n_obs = mean_t_test(cw, bandwidth, alternative="greater")
            test_rows.append(
                {
                    "test": "Clark-West",
                    "horizon": horizon,
                    "rv_proxy": proxy_label,
                    "loss_type": "adjusted_MSPE",
                    "model": model,
                    "statistic": stat,
                    "p_value": p_value,
                    "n": n_obs,
                    "bandwidth": bandwidth,
                    "null": "No MSPE improvement over baseline",
                    "alternative": "one-sided; augmented model improves MSPE",
                    "included_mcs": np.nan,
                    "method": "Newey-West" if bandwidth > 0 else "standard",
                }
            )

        mcs_losses, _, _ = loss_matrix(horizon, proxy_label, "HRMSE")
        included, pvals, mcs_method = run_mcs(mcs_losses, horizon, alpha=0.10)
        for model in MODEL_ORDER:
            test_rows.append(
                {
                    "test": "Model Confidence Set",
                    "horizon": horizon,
                    "rv_proxy": proxy_label,
                    "loss_type": "HRMSE",
                    "model": model,
                    "statistic": np.nan,
                    "p_value": pvals.get(model, np.nan),
                    "n": int(len(mcs_losses)),
                    "bandwidth": np.nan,
                    "null": "Model is in the superior set at alpha=0.10",
                    "alternative": "excluded by iterative MCS",
                    "included_mcs": model in included,
                    "method": mcs_method,
                }
            )

comparison_tests = pd.DataFrame(test_rows)
comparison_tests.to_csv(COMPARISON_TESTS_PATH, index=False)
print(f"Saved comparison tests to {COMPARISON_TESTS_PATH}")

dm_summary = comparison_tests.query("test == 'Diebold-Mariano' and rv_proxy == 'Parkinson'")
display(Markdown("### Diebold-Mariano vs baseline: Parkinson"))
display(dm_summary.pivot_table(index="model", columns=["horizon", "loss_type"], values=["statistic", "p_value"]).style.format("{:.4f}"))

cw_summary = comparison_tests.query("test == 'Clark-West' and rv_proxy == 'Parkinson'")
display(Markdown("### Clark-West vs baseline: Parkinson"))
display(cw_summary.pivot_table(index="model", columns="horizon", values=["statistic", "p_value"]).style.format("{:.4f}"))

mcs_summary = comparison_tests.query("test == 'Model Confidence Set'")
for proxy_label in rv_proxy_map:
    display(Markdown(f"### MCS survivors at 90% level: {proxy_label}"))
    for horizon in HORIZONS:
        survivors = mcs_summary.query(
            "rv_proxy == @proxy_label and horizon == @horizon and included_mcs == True"
        )["model"].tolist()
        print(f"h={horizon}: {', '.join(survivors)}")


## Section 6 - Interpretation and Headline Results

The final cell generates a concise prose interpretation from the saved loss and comparison-test artifacts. It is intentionally data-driven: if sentiment does not help, the text should say so.


In [ ]:
loss_summary = pd.read_csv(LOSSES_CSV_PATH, header=[0, 1, 2], index_col=0)
loss_summary.columns = pd.MultiIndex.from_tuples(
    [(int(h), proxy, loss) for h, proxy, loss in loss_summary.columns],
    names=["horizon", "rv_proxy", "loss"],
)
comparison_tests = pd.read_csv(COMPARISON_TESTS_PATH)
lead_lag_df = pd.read_csv(LEAD_LAG_CSV_PATH)

lines = ["# Section 6 - Interpretation and Headline Results", ""]

lines.append("1. Headline Parkinson HRMSE winners:")
for horizon in HORIZONS:
    col = (horizon, "Parkinson", "HRMSE")
    winner = loss_summary[col].idxmin()
    value = loss_summary.loc[winner, col]
    lines.append(f"   - h={horizon}: {winner} has the lowest HRMSE ({value:.4f}).")

lines.append("")
lines.append("2. Significance versus the GARCH(1,1) baseline:")
for horizon in HORIZONS:
    dm = comparison_tests.query(
        "test == 'Diebold-Mariano' and rv_proxy == 'Parkinson' and horizon == @horizon and p_value < 0.05"
    )
    if dm.empty:
        lines.append(f"   - DM h={horizon}: no augmented model significantly beats the baseline at 5%.")
    else:
        desc = ", ".join(f"{row.model} ({row.loss_type}, p={row.p_value:.3f})" for row in dm.itertuples())
        lines.append(f"   - DM h={horizon}: significant augmented improvements are {desc}.")

    cw = comparison_tests.query(
        "test == 'Clark-West' and rv_proxy == 'Parkinson' and horizon == @horizon and p_value < 0.05"
    )
    if cw.empty:
        lines.append(f"   - CW h={horizon}: no augmented model significantly improves adjusted MSPE at 5%.")
    else:
        desc = ", ".join(f"{row.model} (p={row.p_value:.3f})" for row in cw.itertuples())
        lines.append(f"   - CW h={horizon}: significant augmented improvements are {desc}.")

lines.append("")
lines.append("3. MCS survivors at the 90% level against Parkinson HRMSE loss:")
for horizon in HORIZONS:
    survivors = comparison_tests.query(
        "test == 'Model Confidence Set' and rv_proxy == 'Parkinson' and horizon == @horizon and included_mcs == True"
    )["model"].tolist()
    lines.append(f"   - h={horizon}: {', '.join(survivors) if survivors else 'none'}")

lines.append("")
lines.append("4. Slow-moving-sentiment hypothesis:")
baseline_hrmse = {
    h: float(loss_summary.loc[BASELINE_MODEL, (h, "Parkinson", "HRMSE")])
    for h in HORIZONS
}
best_aug_hrmse = {}
for h in HORIZONS:
    col = (h, "Parkinson", "HRMSE")
    aug_values = loss_summary.loc[[m for m in MODEL_ORDER if m != BASELINE_MODEL], col]
    best_aug_hrmse[h] = float(aug_values.min())
improvement = {
    h: (baseline_hrmse[h] - best_aug_hrmse[h]) / baseline_hrmse[h]
    for h in HORIZONS
}
imp_text = ", ".join(f"h={h}: {100 * improvement[h]:.2f}%" for h in HORIZONS)
longer_better = (improvement[5] > improvement[1]) and (improvement[22] > improvement[1])
if longer_better and (improvement[5] > 0 or improvement[22] > 0):
    lines.append(f"   - Best augmented HRMSE improvement over baseline is {imp_text}. This pattern is consistent with sentiment helping more beyond h=1.")
elif max(improvement.values()) <= 0:
    lines.append(f"   - Best augmented HRMSE improvement over baseline is {imp_text}. The OOS losses do not support the hypothesis.")
else:
    lines.append(f"   - Best augmented HRMSE improvement over baseline is {imp_text}. The evidence is mixed rather than a clean longer-horizon pattern.")

lines.append("")
lines.append("5. Lead-lag diagnostic consistency:")
peak_summaries = []
for series_name, group in lead_lag_df.groupby("series", sort=False):
    peak = group.loc[group["correlation"].abs().idxmax()]
    peak_summaries.append(f"{series_name} peaks at k={int(peak['lag']):+d}")
peak_text = "; ".join(peak_summaries)
if longer_better:
    lines.append(f"   - The diagnostic says {peak_text}. Compare the sign of those peak lags with the longer-horizon gains above; positive peak lags are the most consistent with the slow-moving-sentiment story.")
else:
    lines.append(f"   - The diagnostic says {peak_text}. Since the OOS evidence is not a clean longer-horizon improvement, the diagnostic should be read as context rather than confirmation.")

runtime_minutes = (time.perf_counter() - NOTEBOOK_START) / 60.0
lines.append("")
lines.append(f"Total runtime for this run: {runtime_minutes:.2f} minutes.")

headline_text = "\n".join(lines)
HEADLINE_MD_PATH.write_text(headline_text, encoding="utf-8")
display(Markdown(headline_text))
print(f"Saved headline interpretation to {HEADLINE_MD_PATH}")
